In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device: ", device)
if torch.cuda.is_available():
  print("GPU name: ", torch.cuda.get_device_name(0))

using device:  cuda
GPU name:  Tesla T4


In [18]:
# manual regression with autograd
# y = wx+b
x = torch.tensor([[1.0],[2.0],[3.0],[4.0]])
y_true = torch.tensor([[3.0],[5.0],[7.0],[9.0]])

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
learning_rate = 0.01

for epoch in range(100):
  y_pred = x * w + b
  loss = torch.mean((y_pred - y_true) ** 2)
  loss.backward()

  with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate* b.grad

    w.grad.zero_()
    b.grad.zero_()
  if epoch % 10 == 0:
    print(f"Epoch {epoch}, loss: {loss.item()}")
print("Learned weight:", w.item())
print("Learned bias:", b.item())

Epoch 0, loss: 37.96833419799805
Epoch 10, loss: 0.9957153797149658
Epoch 20, loss: 0.038365960121154785
Epoch 30, loss: 0.012863515876233578
Epoch 40, loss: 0.011512779630720615
Epoch 50, loss: 0.010827085003256798
Epoch 60, loss: 0.010196475312113762
Epoch 70, loss: 0.009603000245988369
Epoch 80, loss: 0.009044062346220016
Epoch 90, loss: 0.008517635986208916
Learned weight: 2.0745413303375244
Learned bias: 0.7808389663696289


In [23]:
# nn.module
# All pytorch neural networks are written in nn.Module
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.linear = nn.Linear(1,1)

  def forward(self, x):
    output = self.linear(x)
    return output
model = LinearRegressionModel()
print(model)


LinearRegressionModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [24]:
# Training nn.Module Model
x = torch.tensor([[1.0],[2.0],[3.0],[4.0]])
y_true = torch.tensor([[3.0],[5.0],[7.0],[9.0]])



criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)

for epoch in range(100):
  y_pred = model(x)

  loss = criterion(y_pred, y_true)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 10 == 0:
    print(f"Epoch {epoch}, loss: {loss.item()}")
print("Learned weight:", w.item())
print("Learned bias:", b.item())


Epoch 0, loss: 27.890106201171875
Epoch 10, loss: 0.7354400157928467
Epoch 20, loss: 0.03207704424858093
Epoch 30, loss: 0.013119780458509922
Epoch 40, loss: 0.011913991533219814
Epoch 50, loss: 0.011209099553525448
Epoch 60, loss: 0.01055637001991272
Epoch 70, loss: 0.009941940195858479
Epoch 80, loss: 0.009363275952637196
Epoch 90, loss: 0.008818275295197964
Learned weight: 2.0745413303375244
Learned bias: 0.7808389663696289


In [28]:
# Linear Layer
# Input shape: batch_size x input_features
# Output shape: batch_size x output_features

linear_layer = nn.Linear(in_features=4, out_features=3)

x = torch.randn(5, 4)

output = linear_layer(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([5, 4])
Output shape: torch.Size([5, 3])


In [31]:
# Activation functions
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

relu_output = F.relu(x)
sigmoid_output = torch.sigmoid(x)
tanh_output = torch.tanh(x)

print("Input:", x)
print("RELU:", relu_output)
print("Sigmoid:", sigmoid_output)
print("Tanh:", tanh_output)

Input: tensor([-2., -1.,  0.,  1.,  2.])
RELU: tensor([0., 0., 0., 1., 2.])
Sigmoid: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Tanh: tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])


In [34]:
# softmax
# converts logits into probabilities

logits = torch.tensor([[2.0, 1.0, 0.1]])
probabilities = F.softmax(logits, dim=1)

print("Logits:", logits)
print("Probabilities:", probabilities)
print("Sum:", probabilities.sum())

Logits: tensor([[2.0000, 1.0000, 0.1000]])
Probabilities: tensor([[0.6590, 0.2424, 0.0986]])
Sum: tensor(1.0000)


In [37]:
# CrossEntropyLoss
# Important:
# It expects raw logits, not softmax probabilities

logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.5, 2.5, 0.3]
])

targets = torch.tensor([0,1])
criterion = nn.CrossEntropyLoss()

loss = criterion(logits, targets)

print("Loss:", loss.item())

Loss: 0.31853973865509033


In [38]:
# Custom Dataset
from torch.utils.data import Dataset, DataLoader

class SimpleDataset(Dataset):
  def __init__(self):
    self.X = torch.randn(100,2)
    self.y = (self.X[:,0] + self.X[:,1] > 0).long()

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

dataset = SimpleDataset()
print("Number of smaples:", len(dataset))

sample_x, sample_y = dataset[0]

print("Sample input:", sample_x)
print("Sample output:", sample_y)

Number of smaples: 100
Sample input: tensor([ 1.0623, -0.4311])
Sample output: tensor(1)


In [41]:
# DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

for batch_x, batch_y in dataloader:
  print("Batch input shape:", batch_x.shape)
  print("Batch output shape:", batch_y.shape)


Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([16, 2])
Batch output shape: torch.Size([16])
Batch input shape: torch.Size([4, 2])
Batch output shape: torch.Size([4])


In [42]:
# MLP for Binary Classification

class MLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, num_classes):
    super().__init__()

    self.fc1 = nn.Linear(input_dim, hidden_dim)
    self.fc2 = nn.Linear(hidden_dim, hidden_dim)
    self.fc3 = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    x = self.fc1(x)
    x = F.relu(x)

    x = self.fc2(x)
    x = F.relu(x)

    x = self.fc3(x)

    return x

model = MLP(input_dim=2, hidden_dim=32, num_classes=2).to(device)
print(model)

MLP(
  (fc1): Linear(in_features=2, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=2, bias=True)
)


In [45]:
# Traninig MLP

dataset = SimpleDataset()
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

model = MLP(input_dim=2, hidden_dim=32, num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
  model.train()

  total_loss = 0
  correct = 0
  total = 0

  for batch_x, batch_y in dataloader:
    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    logits = model(batch_x)

    loss = criterion(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = torch.argmax(logits, dim=1)
    correct += (predictions == batch_y).sum().item()
    total += batch_y.size(0)

  accuracy = correct/total

  print(f"Epoch{epoch+1}, Loss:{total_loss}, Accuracy:{accuracy}")


Epoch1, Loss:4.4043291211128235, Accuracy:0.52
Epoch2, Loss:3.9987872540950775, Accuracy:0.64
Epoch3, Loss:3.7247401773929596, Accuracy:0.81
Epoch4, Loss:3.295385330915451, Accuracy:0.91
Epoch5, Loss:3.0405347049236298, Accuracy:0.96
Epoch6, Loss:2.7037136554718018, Accuracy:0.96
Epoch7, Loss:2.469862252473831, Accuracy:0.96
Epoch8, Loss:2.164121240377426, Accuracy:0.97
Epoch9, Loss:1.8510378897190094, Accuracy:0.97
Epoch10, Loss:1.6562087088823318, Accuracy:0.98
Epoch11, Loss:1.5231812596321106, Accuracy:0.98
Epoch12, Loss:1.3932953104376793, Accuracy:0.98
Epoch13, Loss:1.252065360546112, Accuracy:0.98
Epoch14, Loss:1.0884135589003563, Accuracy:0.98
Epoch15, Loss:1.023003987967968, Accuracy:0.98
Epoch16, Loss:0.9930349439382553, Accuracy:0.98
Epoch17, Loss:0.9912444427609444, Accuracy:0.98
Epoch18, Loss:0.7766508013010025, Accuracy:0.99
Epoch19, Loss:0.7603653445839882, Accuracy:1.0
Epoch20, Loss:0.782384280115366, Accuracy:1.0


In [1]:
# Install Git if not already available
!apt-get install git

# Configure your GitHub identity (replace with your details)
!git config --global user.email "kottanapradeep123@gmail.com"
!git config --global user.name "pradeep-kottana"


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
!git clone https://github.com/pradeep-kottana/Mathematical-foundations-to-Generative-AI.git
%cd Mathematical-foundations-to-Generative-AI


Cloning into 'Mathematical-foundations-to-Generative-AI'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Mathematical-foundations-to-Generative-AI


In [3]:
!mkdir Gen-AI-files


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/My Drive/Colab Notebooks/Linear_regression_model_and_MLP.ipynb" "/content/Mathematical-foundations-to-Generative-AI/Gen-AI-files/"

In [10]:
%cd Mathematical-foundations-to-Generative-AI
!git add Gen-AI-files/Linear_regression_model_and_MLP.ipynb
!git commit -m "Added Linear_regression_model_and_MLP.ipynb to Gen-AI-files folder"


[Errno 2] No such file or directory: 'Mathematical-foundations-to-Generative-AI'
/content/Mathematical-foundations-to-Generative-AI
[main d98a7ad] Added Linear_regression_model_and_MLP.ipynb to Gen-AI-files folder
 1 file changed, 1 insertion(+)
 create mode 100644 Gen-AI-files/Linear_regression_model_and_MLP.ipynb
